# 04. Amplitude clipping and phasor normalisation

This notebook isolates the magnitude handling inside interferogram formation: the secondary amplitude is hard-clipped at `max_amplitude_clip`, while the phasor is divided by its own magnitude (plus 1e-30) so it carries phase only. Together these set the magnitude of the output interferogram. The notebook shows the clip acting on the amplitude distribution and confirms the normalisation leaves phase untouched.

**Modules exercised:** pipelines.processing_pipeline.interferogram (amplitude clip and phasor normalisation), configuration.processing_config (TomogramConfiguration.max_amplitude_clip)

In [ ]:
import sys
from pathlib import Path

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except ImportError:
    torch = None

RNG = np.random.default_rng(0)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : False,
    "image.cmap"     : "viridis",
})

print("repo root:", repo_root)


## A scene with bright outliers

We build amplitudes with a heavy tail so the clip has a visible effect. The phase is an arbitrary smooth field whose preservation we will check.

In [ ]:
n_az, n_rg = 40, 60
max_amplitude_clip = 1.25

base_amp = 0.6 + 0.3 * RNG.standard_normal((n_az, n_rg))
base_amp = np.clip(base_amp, 0.0, None)
base_amp[10:14, 20:24] = 3.0
base_amp[30, 50]       = 5.0

az = np.arange(n_az)[:, None]; rg = np.arange(n_rg)[None, :]
phase_field = 0.15 * az + 0.10 * rg
secondary   = (base_amp * np.exp(1.0j * phase_field)).astype(np.complex64)
primary     = np.ones((n_az, n_rg), dtype=np.complex64)
print('amplitude max before clip:', float(base_amp.max()))

## Apply the clip and normalisation

Same arithmetic as the builder. We record the phase of the raw phasor and of the normalised phasor to confirm normalisation does not move the phase.

In [ ]:
secondary_amplitude = np.clip(np.abs(secondary), 0.0, max_amplitude_clip)
phasor_raw          = primary * np.conj(secondary)
phasor_norm         = phasor_raw / (np.abs(phasor_raw) + 1e-30)
interferogram       = secondary_amplitude * phasor_norm

phase_drift = np.max(np.abs(np.angle(phasor_norm) - np.angle(phasor_raw)))
print('max phase change from normalisation:', float(phase_drift))
print('|phasor_norm| mean (should be ~1):', float(np.abs(phasor_norm).mean()))
print('|interferogram| max (<= clip):', float(np.abs(interferogram).max()))

## Amplitude distribution before and after clipping

The histogram should show all mass above the clip value collapsed onto the clip threshold.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.hist(np.abs(secondary).ravel(), bins=40, alpha=0.6, label='raw |secondary|')
ax.hist(secondary_amplitude.ravel(), bins=40, alpha=0.6, label='clipped amplitude')
ax.axvline(max_amplitude_clip, color='k', lw=1.0, ls='--', label='clip threshold')
ax.set_xlabel('amplitude'); ax.set_ylabel('count')
ax.set_title('Effect of max_amplitude_clip')
ax.legend()
fig.tight_layout()
plt.show()

## Magnitude image and preserved phase

The interferogram magnitude should equal the clipped amplitude, and its phase should equal the original phasor phase.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
im0 = axes[0].imshow(np.abs(interferogram), aspect='auto', origin='lower', vmax=max_amplitude_clip)
axes[0].set_title('|interferogram| (= clipped amp)')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02)
im1 = axes[1].imshow(np.angle(interferogram), cmap='twilight', vmin=-np.pi, vmax=np.pi, aspect='auto', origin='lower')
axes[1].set_title('interferogram phase')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02)
for ax in axes:
    ax.set_xlabel('range'); ax.set_ylabel('azimuth')
fig.tight_layout()
plt.show()

## Expected visual outcome

The histogram shows raw amplitudes extending past the threshold while the clipped amplitudes pile up exactly at it. The reported phase change from normalisation is essentially zero and `|phasor_norm|` is one, confirming the normalisation strips magnitude without disturbing phase, and the interferogram magnitude is bounded by the clip.